# Instability Attribution — RQ3 (L5B15)

Decomposes observed ranking instability (notebook 07) into three sources, each measured with a
metric matched to the underlying object:

- **δ_d (distribution shift)** — propensity-score distributions differ between sub-populations,
  measured by the **Kolmogorov–Smirnov statistic on propensity**. KS is bin-free and operates
  directly on the empirical CDFs, which is essential because Pega ADM's isotonic calibration
  produces a step-function propensity (clustered on a small set of recurring values), making
  PSI bin-sensitive. PSI diagnostics are retained in Appendix A §A1.
- **δ_m (model churn)** — Pega ADM's online Naive Bayes updates between splits. Measured by
  **Jaccard overlap on the set of `modelVersion` IDs** active in each split. Low overlap = high
  churn. This is a more faithful proxy than "KS on AUC" because `modelPerformance` is essentially
  a categorical label of which model snapshot scored each event (heavy ties, mixture-of-snapshots),
  not a continuous performance measurement. Jaccard answers the actual question directly:
  *did the same model snapshots score both halves of the split?* `J_v` measures
  **snapshot turnover** rather than parameter shift directly — UUIDs rotate on every micro-update
  of the online NB regardless of how much the parameters moved. The RQ3 conclusion is robust
  to this because same-window splits share the same active snapshot pool by construction; any
  reasonable churn proxy returns ≈ 0 there.
- **δ_e (explainer sensitivity)** — the explanation method's own variance, decomposed into
  **two complementary mechanisms** measured against SHAP as the deterministic reference
  (TreeSHAP is deterministic given a fixed surrogate):
  - **δ_e^DT** = `ρ_SHAP − ρ_DT` — DT *refit sensitivity*: the tree structure is re-fitted
    independently on each split, so this captures refit variance.
  - **δ_e^LIME** = `ρ_SHAP − ρ_LIME` — LIME *sampling sensitivity*: LIME shares the
    fixed-surrogate property with SHAP, so any gap here is local-perturbation noise,
    not refit.

  Both are reported as descriptive evidence; neither is a classifier input.

**Classification — effect-size only, no p-values.** With n in the thousands per split, p-values
are sample-size driven (a KS of 0.04 reaches p = 0.000 with n ≈ 23k). The KS statistic itself is
the population-level effect size at this sample size, so we threshold on effect size directly:

- δ_d significant ⇔ **KS_propensity ≥ 0.10**
- δ_m significant ⇔ **Jaccard(modelVersion) ≤ 0.10**
  (see Appendix A §A4: J_v values are bimodal — temporal splits near 0, same-window splits
  near 0.5 — so the threshold sits in the empty gap and classifications are invariant for
  τ ∈ [0.05, 0.40])

The classification rule uses only KS_prop and J_v. δ_e^DT and δ_e^LIME are reported alongside
the classification as descriptive evidence: large values validate the *Explainer* label;
small values qualify a *Dist. shift / Model churn / Both* label as also clean of explainer effects.

**Why not PSI.** PSI on propensity is bin-sensitive on Pega's quantised scores (Appendix A §A1).
PSI on AUC is stable (§A2) but we're not using AUC for δ_m any more. PSI tables are retained in
Appendix A for transparency.

This notebook covers the **primary L5B15 model only**. RQ4 replication for CLUG / BookingDotCom /
Cartrawler is in `08b_attribution_replication.ipynb`.

**Requires:** notebook 07 stability summary at `data/artifacts/L5B15/stability_summary.json`.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
from pathlib import Path

PRIMARY_VARIANT = "L5B15"

PROCESSED_DIR  = Path("../data/processed")
ARTIFACT_DIR   = Path("../data/artifacts")

TEMPORAL_SPLIT = "2026-04-27"   # same split as notebook 07
MIN_ROUTE_OBS  = 400            # lowered from 500 after modelTechnique=="0.0"
                                # filter halved row counts (matches notebook 07)
TOP_N_ROUTES   = 4
TOP_FEATURES_N = 5              # numeric features to show in per-feature PSI table
PSI_BINS       = 5              # bins for PSI; 5 avoids empty-bin inflation on quantised propensity scores

print(f"Primary variant : {PRIMARY_VARIANT}")
print(f"PSI bins        : {PSI_BINS}")

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
from my_project.features import VARIANT_FEATURES
from my_project.metrics import psi
from my_project.surrogate import build_feature_matrix

print("Imports OK")

In [ ]:
# ── Load primary variant data ──────────────────────────────────────────────
import json as _json
import pandas as _pd

_lug = _pd.read_parquet(PROCESSED_DIR / "luggage_email_outbound.parquet")
_tp_path = PROCESSED_DIR / "thirdparty_email_outbound.parquet"
df_raw = _pd.concat(
    [_lug, _pd.read_parquet(_tp_path)] if _tp_path.exists() else [_lug],
    ignore_index=True,
)
del _lug, _pd, _tp_path

# Restrict to the production-decision technique (label "0.0"). Every scoring
# event is evaluated by two parallel techniques; analysis is on the model
# that actually drives offer decisions, matching what 05_surrogate_fit
# fitted. See 03_eda §6.10.
df_raw = df_raw[df_raw["modelTechnique"] == "0.0"].reset_index(drop=True)

df_raw["pxDecisionTime"] = pd.to_datetime(df_raw["pxDecisionTime"], utc=True, errors="coerce")

df = df_raw[df_raw["pyName"] == PRIMARY_VARIANT].reset_index(drop=True)

cfg = VARIANT_FEATURES[PRIMARY_VARIANT]

# Use the EXACT training feature set + order saved by 05_surrogate_fit, not
# cfg.features. Keeps X consistent with what the saved CatBoost was trained on.
saved_feature_cols = _json.loads(
    (ARTIFACT_DIR / PRIMARY_VARIANT / "feature_cols.json").read_text()
)
X, y, cat_cols, num_cols = build_feature_matrix(df, saved_feature_cols, cfg.numeric)

meta = df[["pxDecisionTime",
           "modelEvidence", "modelPerformance", "modelVersion",
           "CustBookedFlight.FlightData.DepartureAirport",
           "CustBookedFlight.FlightData.DestinationAirport"]].copy()
meta["route"] = (meta["CustBookedFlight.FlightData.DepartureAirport"].astype(str)
                 + "->" + meta["CustBookedFlight.FlightData.DestinationAirport"].astype(str))

# Temporal split
split_date = pd.Timestamp(TEMPORAL_SPLIT, tz="UTC")
mask_early = meta["pxDecisionTime"] <= split_date
mask_late  = meta["pxDecisionTime"] >  split_date

# Route subgroups (top N by volume)
route_counts = meta["route"].value_counts()
top_routes   = route_counts[route_counts >= MIN_ROUTE_OBS].head(TOP_N_ROUTES).index.tolist()
route_masks  = {r: meta["route"] == r for r in top_routes}

# Load SHAP importances to identify most important features
shap_imp = pd.read_json(ARTIFACT_DIR / PRIMARY_VARIANT / "shap_importances.json", typ="series")\
             .sort_values(ascending=False)

print(f"Dataset: {len(df):,} rows  |  {X.shape[1]} features")
print(f"Early: {mask_early.sum():,}  |  Late: {mask_late.sum():,}")
print(f"Top routes: {top_routes}")

## §18 Model Churn (δ_m)

Pega ADM's Naive Bayes updates online with each new customer interaction.  
`modelEvidence` tracks the cumulative number of observations used to fit the model.  
`modelPerformance` is the model's AUC at each scoring event.  
Growth in `modelEvidence` indicates the model has absorbed new data between splits,
potentially shifting feature weights and propensity outputs for identical inputs.

In [ ]:
# ── Weekly model evidence and performance ──────────────────────────────────
meta_sorted = meta.sort_values("pxDecisionTime").copy()
meta_sorted["week"] = meta_sorted["pxDecisionTime"].dt.to_period("W").astype(str)

weekly = (
    meta_sorted.groupby("week")
    .agg(
        evidence_mean=("modelEvidence",   "mean"),
        performance_mean=("modelPerformance", "mean"),
        n=("modelVersion", "count"),
        n_versions=("modelVersion", "nunique"),
    )
    .reset_index()
)

split_week_idx = (weekly["week"] <= TEMPORAL_SPLIT).sum() - 1

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

axes[0].plot(range(len(weekly)), weekly["evidence_mean"] / 1e6, marker="o", color="steelblue")
axes[0].axvline(x=split_week_idx, color="gray", linestyle="--", linewidth=0.9, label="split")
axes[0].set_xticks(range(len(weekly)))
axes[0].set_xticklabels(weekly["week"], rotation=30, ha="right", fontsize=8)
axes[0].set_title("Mean modelEvidence per week", fontsize=10)
axes[0].set_ylabel("Evidence (millions)")
axes[0].legend()

axes[1].plot(range(len(weekly)), weekly["performance_mean"], marker="o", color="seagreen")
axes[1].axvline(x=split_week_idx, color="gray", linestyle="--", linewidth=0.9, label="split")
axes[1].set_xticks(range(len(weekly)))
axes[1].set_xticklabels(weekly["week"], rotation=30, ha="right", fontsize=8)
axes[1].set_ylim(0.5, 1.0)
axes[1].set_title("Mean modelPerformance (AUC) per week", fontsize=10)
axes[1].set_ylabel("AUC")
axes[1].legend()

fig.suptitle(f"Pega ADM model churn — {PRIMARY_VARIANT}", fontsize=11)
plt.tight_layout()
plt.show()

# Compute Δ metrics
ev_early  = meta.loc[mask_early, "modelEvidence"].mean()
ev_late   = meta.loc[mask_late,  "modelEvidence"].mean()
pf_early  = meta.loc[mask_early, "modelPerformance"].mean()
pf_late   = meta.loc[mask_late,  "modelPerformance"].mean()
delta_ev  = (ev_late - ev_early) / ev_early * 100
delta_pf  = pf_late - pf_early

print(f"modelEvidence:    early={ev_early:,.0f}  late={ev_late:,.0f}  delta={delta_ev:+.1f}%")
print(f"modelPerformance: early={pf_early:.4f}   late={pf_late:.4f}   delta={delta_pf:+.4f}")

In [ ]:
# ── modelVersion diversity ─────────────────────────────────────────────────
# Each unique modelVersion UUID reflects a Pega model snapshot.
# High diversity indicates frequent weight updates between scoring events.
print("modelVersion diversity:")
print(f"  Total unique versions : {meta['modelVersion'].nunique():,}")
print(f"  Early unique versions : {meta.loc[mask_early, 'modelVersion'].nunique():,}")
print(f"  Late  unique versions : {meta.loc[mask_late,  'modelVersion'].nunique():,}")
print(f"  Mean rows per version : {len(meta) / meta['modelVersion'].nunique():.1f}")
print()
print("Weekly version diversity:")
display(
    weekly[["week", "n", "n_versions"]]
    .rename(columns={"n": "rows", "n_versions": "unique_versions"})
    .set_index("week")
)

## §19 Explainer Sensitivity (δ_e)

SHAP serves as the deterministic reference: TreeSHAP is exact given the fixed CatBoost surrogate
(fitted once on the full training set, applied identically to both splits), so SHAP's ranking
shift between splits reflects only data drift and model churn with no explainer-side contribution.

DT and LIME are compared against this reference, probing **two distinct mechanisms** of
explainer sensitivity:

- **δ_e^DT = ρ_SHAP − ρ_DT** — DT *refit sensitivity*. The DT surrogate is re-fitted on each
  split (cross-validated depth + structure); any gap to SHAP captures refit-induced variance.
- **δ_e^LIME = ρ_SHAP − ρ_LIME** — LIME *sampling sensitivity*. LIME shares the fixed-surrogate
  property with SHAP, so any gap here is local-perturbation-sampling noise, not refit.

Both gaps are positive when the corresponding explainer is less stable than SHAP. They are
complementary facets of the same construct (sensitivity of the explanation method to nuisance
variation) and are reported jointly throughout.

In [ ]:
# ── Load stability summary from notebook 07 ────────────────────────────────
stab_path = ARTIFACT_DIR / PRIMARY_VARIANT / "stability_summary.json"
stab_df   = pd.read_json(stab_path)

# Pivot: one row per split, columns = DT / SHAP / LIME
stab_pivot = stab_df.pivot(index="split", columns="method", values="Spearman \u03c1")
stab_pivot.columns.name = None
stab_pivot["delta_rho"] = stab_pivot["SHAP"] - stab_pivot["DT"]

# Sort: temporal splits first (delta_rho desc), then route pairs (delta_rho desc)
_is_route = stab_pivot.index.str.contains("->")
stab_pivot = pd.concat([
    stab_pivot[~_is_route].sort_values("delta_rho", ascending=False),
    stab_pivot[_is_route].sort_values("delta_rho", ascending=False),
])

display_pivot = stab_pivot.rename(columns={
    "DT": "rho_DT", "SHAP": "rho_SHAP", "LIME": "rho_LIME", "delta_rho": "delta_rho (SHAP-DT)"
})

print("Explainer sensitivity per split (Spearman rho):")
display(
    display_pivot.style
    .format("{:.4f}")
    .background_gradient(subset=["rho_DT", "rho_SHAP", "rho_LIME"], cmap="RdYlGn", vmin=0.4, vmax=1.0)
    .background_gradient(subset=["delta_rho (SHAP-DT)"], cmap="YlOrRd", vmin=0, vmax=0.5)
)

# Bar chart comparing all three methods per split
fig, ax = plt.subplots(figsize=(9, 3.5))
x = range(len(stab_pivot))
w = 0.25
ax.bar([i - w for i in x], stab_pivot["DT"],   width=w, label="DT",   color="darkorange", alpha=0.82)
ax.bar([i      for i in x], stab_pivot["SHAP"], width=w, label="SHAP", color="steelblue",  alpha=0.82)
ax.bar([i + w for i in x], stab_pivot["LIME"],  width=w, label="LIME", color="seagreen",   alpha=0.82)
ax.set_xticks(list(x))
ax.set_xticklabels(stab_pivot.index, rotation=15, ha="right", fontsize=8)
ax.set_ylim(0, 1.05)
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.7)
ax.set_ylabel("Spearman rho")
ax.set_title("Spearman rho per split and explainer method", fontsize=10)
ax.legend()
plt.tight_layout()
plt.show()

## §20 Attribution Summary

Unified table linking each split to the three instability sources.

- **KS_prop (δ_d)** — KS on the propensity distribution; significant if **KS ≥ 0.10**
- **J_v (δ_m)** — Jaccard overlap on `modelVersion` sets; significant if **J_v ≤ 0.10**
  (see Appendix A §A4 for the sensitivity analysis behind this threshold)
- **δ_e^DT** — `ρ_SHAP − ρ_DT`; DT refit sensitivity
- **δ_e^LIME** — `ρ_SHAP − ρ_LIME`; LIME sampling sensitivity

Both explainer-gap columns are descriptive evidence, not classifier inputs. The Primary source
column uses only KS_prop and J_v; the explainer gaps validate (when large) or qualify
(when small) the classification.

Effect-size only: p-values are not used in the classifier (with n in the thousands, p-values are
sample-size driven and uninformative — e.g. fr-FR vs nl-NL has KS_AUC = 0.04 but p = 0.000).
PSI-based diagnostics are in Appendix A.

In [ ]:
# ── δ_d via KS on propensity, δ_m via Jaccard on modelVersion ─────────────
# Effect-size-only classification. p-values are not consulted because at this
# sample size they are sample-size driven (a trivially small KS reaches p<0.05
# given n in the thousands). The KS statistic *is* the effect size at this n.
#
# δ_m moves from "KS on modelPerformance (AUC)" to Jaccard on the modelVersion
# set. modelPerformance is effectively a categorical snapshot identifier with
# heavy ties; Jaccard directly answers "how much snapshot overlap is there?"
# without forcing categorical mixtures into a continuous-KS frame.
#
# δ_e is decomposed into two mechanisms — δ_e^DT (refit) and δ_e^LIME (sampling) —
# both measured against SHAP as the deterministic reference.
from scipy.stats import ks_2samp as _ks2

def _ks_stat(s_a, s_b):
    stat, _p = _ks2(s_a, s_b)
    return round(stat, 4)

def _jaccard(set_a, set_b):
    set_a, set_b = set(set_a), set(set_b)
    union = set_a | set_b
    return 1.0 if not union else round(len(set_a & set_b) / len(union), 4)

ks_prop_lookup: dict[str, float] = {}
jaccard_v_lookup: dict[str, float] = {}

# Temporal
ks_prop_lookup[f"{PRIMARY_VARIANT} temporal"] = _ks_stat(y[mask_early], y[mask_late])
jaccard_v_lookup[f"{PRIMARY_VARIANT} temporal"] = _jaccard(
    meta.loc[mask_early, "modelVersion"].dropna(),
    meta.loc[mask_late,  "modelVersion"].dropna(),
)

# Route pairs
for i in range(len(top_routes) - 1):
    r1, r2 = top_routes[i], top_routes[i + 1]
    k = f"{r1} vs {r2}"
    ks_prop_lookup[k]    = _ks_stat(y[route_masks[r1]], y[route_masks[r2]])
    jaccard_v_lookup[k]  = _jaccard(
        meta.loc[route_masks[r1], "modelVersion"].dropna(),
        meta.loc[route_masks[r2], "modelVersion"].dropna(),
    )

# Culture
culture_col = "CustBookedFlight.BookingData.CultureCode"
_pc1 = df.loc[df[culture_col] == "fr-FR", "propensity"].dropna()
_pc2 = df.loc[df[culture_col] == "nl-NL", "propensity"].dropna()
ks_prop_lookup["fr-FR vs nl-NL"]   = _ks_stat(_pc1, _pc2)
jaccard_v_lookup["fr-FR vs nl-NL"] = _jaccard(
    df.loc[df[culture_col] == "fr-FR", "modelVersion"].dropna(),
    df.loc[df[culture_col] == "nl-NL", "modelVersion"].dropna(),
)


# ── Classify by primary instability source (effect-size thresholds) ───────
# δ_d significant ⇔ KS_propensity ≥ 0.10
# δ_m significant ⇔ Jaccard(modelVersion) ≤ 0.10  (see Appendix A §A4)
#
# Classifier uses ONLY KS_prop and J_v. δ_e^DT and δ_e^LIME are reported as
# descriptive evidence — they are not classifier inputs.
KS_PROP_THRESHOLD   = 0.10
JACCARD_V_THRESHOLD = 0.10

def _source(ks_d, j_v):
    d_sig = (not pd.isna(ks_d)) and ks_d >= KS_PROP_THRESHOLD
    m_sig = (not pd.isna(j_v))  and j_v  <= JACCARD_V_THRESHOLD
    if not d_sig and not m_sig: return "Explainer"
    if d_sig and not m_sig:     return "Dist. shift"
    if m_sig and not d_sig:     return "Model churn"
    return "Both"


# ── Assemble attribution table ─────────────────────────────────────────────
# Filter stab_pivot to L5B15 splits (drop any replication-variant rows if the
# loaded summary still contains them — nb 08b owns the variant attribution).
l5b15_splits = [s for s in stab_pivot.index if s in ks_prop_lookup]
stab_pivot   = stab_pivot.loc[l5b15_splits]

attr_rows = []
for split in stab_pivot.index:
    ks_d = ks_prop_lookup[split]
    j_v  = jaccard_v_lookup[split]
    rho_dt   = stab_pivot.loc[split, "DT"]
    rho_shap = stab_pivot.loc[split, "SHAP"]
    rho_lime = stab_pivot.loc[split, "LIME"]
    dpi_dt   = 1 - rho_dt
    dpi_shap = 1 - rho_shap
    dpi_lime = 1 - rho_lime
    delta_e_dt   = round(rho_shap - rho_dt,   4)   # = dpi_dt   - dpi_shap
    delta_e_lime = round(rho_shap - rho_lime, 4)   # = dpi_lime - dpi_shap
    attr_rows.append({
        "split":            split,
        "KS_prop (δ_d)":    ks_d,
        "J_v (δ_m)":        j_v,
        "ρ_DT":             round(rho_dt,   4),
        "ρ_SHAP":           round(rho_shap, 4),
        "ρ_LIME":           round(rho_lime, 4),
        "Δπ_DT":            round(dpi_dt,   4),
        "Δπ_SHAP":          round(dpi_shap, 4),
        "Δπ_LIME":          round(dpi_lime, 4),
        "δ_e^DT":           delta_e_dt,
        "δ_e^LIME":         delta_e_lime,
        "Primary source":   _source(ks_d, j_v),
    })

attr_df = pd.DataFrame(attr_rows).set_index("split")

is_temporal = attr_df.index.str.contains("temporal")
attr_df = pd.concat([
    attr_df[is_temporal].sort_values("δ_e^DT", ascending=False),
    attr_df[~is_temporal].sort_values("δ_e^DT", ascending=False),
])

# ── Style and display ──────────────────────────────────────────────────────
_SOURCE_COLORS = {
    "Explainer":   "background-color: #dff0d8; font-weight: bold",
    "Dist. shift": "background-color: #ffe0b2; font-weight: bold",
    "Model churn": "background-color: #cce5ff; font-weight: bold",
    "Both":        "background-color: #f8d7da; font-weight: bold",
}
def _highlight_source(col):
    return [_SOURCE_COLORS.get(v, "") for v in col]

float_cols = list(attr_df.select_dtypes(float).columns)
print("Attribution summary — effect-size decomposition (L5B15)")
print("  KS_prop (δ_d) = KS on propensity distribution                       (threshold ≥ 0.10)")
print("  J_v     (δ_m) = Jaccard overlap on modelVersion sets                (threshold ≤ 0.10)")
print("  δ_e^DT        = ρ_SHAP − ρ_DT     (DT refit sensitivity, descriptive)")
print("  δ_e^LIME      = ρ_SHAP − ρ_LIME   (LIME sampling sensitivity, descriptive)")
print()
print("  Green  = Explainer       neither external trigger fired")
print("  Orange = Dist. shift     δ_d significant only")
print("  Blue   = Model churn     δ_m significant only")
print("  Red    = Both            both significant")
print()
display(
    attr_df.style
    .format({c: "{:.4f}" for c in float_cols})
    .background_gradient(subset=["KS_prop (δ_d)"],            cmap="YlOrRd",  vmin=0,   vmax=0.5)
    .background_gradient(subset=["J_v (δ_m)"],                cmap="RdYlGn",  vmin=0,   vmax=1.0)
    .background_gradient(subset=["ρ_DT", "ρ_SHAP", "ρ_LIME"], cmap="RdYlGn",  vmin=0.4, vmax=1.0)
    .background_gradient(subset=["Δπ_DT", "Δπ_SHAP", "Δπ_LIME"], cmap="RdYlGn_r", vmin=0, vmax=0.5)
    .background_gradient(subset=["δ_e^DT"],                   cmap="YlOrRd", vmin=0,   vmax=0.5)
    .background_gradient(subset=["δ_e^LIME"],                 cmap="YlOrRd", vmin=0,   vmax=0.5)
    .apply(_highlight_source, subset=["Primary source"])
)
print()
print("Note: classifier uses only KS_prop and J_v. δ_e^DT and δ_e^LIME are reported as")
print("descriptive evidence — a large gap validates the Explainer label; a small gap qualifies")
print("a Dist./Churn/Both label as also clean of explainer effects. p-values intentionally")
print("omitted (at n in the thousands they are sample-size driven). PSI tables are in")
print("Appendix A; the Jaccard-threshold sensitivity analysis is in §A4.")

In [ ]:
# ── Save attribution summary ───────────────────────────────────────────────
out_path = ARTIFACT_DIR / PRIMARY_VARIANT / "attribution_summary.json"
attr_df.reset_index().to_json(out_path, orient="records")
print(f"Saved to {out_path}")

_t = f"{PRIMARY_VARIANT} temporal"
print("\nKey findings (L5B15 temporal split):")
print(f"  KS_prop (δ_d)             : {ks_prop_lookup[_t]:.4f}")
print(f"  Jaccard_v (δ_m)           : {jaccard_v_lookup[_t]:.4f}   "
      f"(unique versions: early={meta.loc[mask_early, 'modelVersion'].nunique():,}, "
      f"late={meta.loc[mask_late, 'modelVersion'].nunique():,})")
print(f"  ΔEvidence                 : {delta_ev:+.1f}%")
print(f"  ΔPerformance              : {delta_pf:+.4f}")
print(f"  Max δ_e^DT                : {attr_df['δ_e^DT'].max():.4f}  ({attr_df['δ_e^DT'].idxmax()})")
print(f"  Min δ_e^DT                : {attr_df['δ_e^DT'].min():.4f}  ({attr_df['δ_e^DT'].idxmin()})")
print(f"  Max δ_e^LIME              : {attr_df['δ_e^LIME'].max():.4f}  ({attr_df['δ_e^LIME'].idxmax()})")
print(f"  Min δ_e^LIME              : {attr_df['δ_e^LIME'].min():.4f}  ({attr_df['δ_e^LIME'].idxmin()})")

## Appendix A — PSI-based diagnostics + Jaccard threshold sensitivity

This appendix retains the original PSI analysis for transparency and adds the sensitivity
analysis that motivates the Jaccard threshold for δ_m.

- **§A0** — Propensity PSI (temporal, route pairs, top features), with histogram
- **§A1** — PSI sensitivity to bin count, **propensity**: PSI grows with bin count → unreliable
- **§A2** — PSI sensitivity to bin count, **AUC**: stable → PSI would have worked for δ_m alone
- **§A3** — Side-by-side comparison of δ_m proxies (AUC PSI vs |Δμ| vs KS)
- **§A4** — **Jaccard threshold sensitivity** for δ_m: why we use `J_v ≤ 0.10`

CSV outputs (consumed by `scripts/build_thesis_tables.py` to produce the appendix LaTeX):
- `psi_bin_sensitivity_propensity.csv` — §A1 matrix
- `psi_bin_sensitivity_auc.csv` — §A2 matrix

## §A0 PSI on propensity (descriptive)

**Population Stability Index (PSI)** quantifies how much a score distribution has shifted between
two populations. Computed with equal-frequency bins on the reference (early / first) split.

PSI thresholds: < 0.10 stable · 0.10–0.25 moderate shift · > 0.25 significant shift.

Reported here for the propensity distribution and the top SHAP-active numeric features. The
sensitivity check in §A1 then shows why these numbers should not be used as the primary δ_d
metric on Pega's quantised propensity output.

In [ ]:
# ── Temporal PSI (propensity scores + top numeric features) ────────────────
psi_temporal, psi_temporal_lbl = psi(y[mask_early], y[mask_late], bins=PSI_BINS)
print(f"Propensity PSI (early vs late): {psi_temporal:.4f}  [{psi_temporal_lbl}]")

# Per-feature PSI for top numeric features
top_num_feats = [f for f in shap_imp.index if f in num_cols][:TOP_FEATURES_N]
feat_psi_rows = []
for feat in top_num_feats:
    vals_e = X.loc[mask_early, feat].dropna()
    vals_l = X.loc[mask_late,  feat].dropna()
    psi_v, psi_lbl = psi(vals_e, vals_l, bins=PSI_BINS)
    parts = feat.split(".")
    feat_psi_rows.append({
        "feature":   ".".join(parts[-3:]),   # last 3 path components — unique across IH features
        "PSI":       round(psi_v, 4),
        "label":     psi_lbl,
    })

feat_psi_df = pd.DataFrame(feat_psi_rows)
print(f"\nPer-feature PSI (top-{TOP_FEATURES_N} numeric by SHAP importance):")
display(
    feat_psi_df.set_index("feature")
    .style
    .format({"PSI": "{:.4f}"})
    .background_gradient(subset=["PSI"], cmap="YlOrRd", vmin=0, vmax=0.25)
)

# Propensity distribution overlay
fig, ax = plt.subplots(figsize=(8, 3))
bins_plot = np.linspace(0, float(np.nanpercentile(y, 99)), 40)
ax.hist(y[mask_early], bins=bins_plot, alpha=0.55, label="Early",  color="steelblue",  density=True)
ax.hist(y[mask_late],  bins=bins_plot, alpha=0.55, label="Late",   color="darkorange", density=True)
ax.set_xlabel("Propensity $\hat{p}$")
ax.set_ylabel("Density")
ax.set_title(
    f"L5B15 propensity — early vs late  (PSI={psi_temporal:.4f}, {psi_temporal_lbl})",
    fontsize=10
)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Route-pair PSI (propensity scores between adjacent route subgroups) ──────
route_masks = {r: meta["route"] == r for r in top_routes}

route_psi_rows = []
for i in range(len(top_routes) - 1):
    r1, r2 = top_routes[i], top_routes[i + 1]
    psi_v, psi_lbl = psi(y[route_masks[r1]], y[route_masks[r2]], bins=PSI_BINS)
    route_psi_rows.append({"split": f"{r1} vs {r2}", "PSI": round(psi_v, 4), "label": psi_lbl})

route_psi_df = pd.DataFrame(route_psi_rows)
print("Route-pair propensity PSI:")
display(
    route_psi_df.set_index("split")
    .style
    .background_gradient(subset=["PSI"], cmap="YlOrRd", vmin=0, vmax=0.25)
    .format({"PSI": "{:.4f}"})
)

In [ ]:
# ── §A1  PSI sensitivity to bin count (propensity) ────────────────────────
# Justifies the choice of PSI_BINS=5.
# We compute propensity PSI across all splits for bins = 3, 4, 5, 6, 7, 10:
#   - If absolute PSI values are stable across bin choices, any value works.
#   - If bins >= 7 inflate PSI (due to Pega's quantised propensity output),
#     the lower values are more reliable.
#   - The ordering of splits by PSI should be stable regardless of bin count.

from scipy.stats import spearmanr as _sp_sens

bin_range = [3, 4, 5, 6, 7, 10]

splits_for_sens = {"L5B15 temporal": (y[mask_early], y[mask_late])}
for i in range(len(top_routes) - 1):
    r1, r2 = top_routes[i], top_routes[i + 1]
    splits_for_sens[f"{r1} vs {r2}"] = (y[route_masks[r1]], y[route_masks[r2]])

# Culture split (added for parity with the main attribution analysis)
_culture_col = "CustBookedFlight.BookingData.CultureCode"
_mask_fr = df[_culture_col] == "fr-FR"
_mask_nl = df[_culture_col] == "nl-NL"
if _mask_fr.any() and _mask_nl.any():
    splits_for_sens["fr-FR vs nl-NL"] = (y[_mask_fr], y[_mask_nl])

sens_rows = []
for b in bin_range:
    row = {"bins": b}
    for split_name, (s_a, s_b) in splits_for_sens.items():
        psi_v, _ = psi(s_a, s_b, bins=b)
        row[split_name] = round(psi_v, 4)
    sens_rows.append(row)

sens_df = pd.DataFrame(sens_rows).set_index("bins")

print("PSI values per split at each bin count:")
display(
    sens_df.style
    .format("{:.4f}")
    .background_gradient(cmap="YlOrRd", vmin=0, vmax=0.30, axis=None)
    .highlight_max(color="#f8d7da", axis=0)
)

# Ordering stability: Spearman rho between PSI rankings at bins=5 vs all others
ref_ranks = sens_df.loc[PSI_BINS].rank()
print(f"Spearman rho of split ordering (vs PSI_BINS={PSI_BINS} reference):")
for b in bin_range:
    rho, _ = _sp_sens(ref_ranks, sens_df.loc[b].rank())
    marker = " <-- chosen" if b == PSI_BINS else ""
    print(f"  bins={b:2d}  ordering rho={rho:.4f}{marker}")

# Plot PSI vs bins for each split
fig, ax = plt.subplots(figsize=(8, 3.5))
for split_name in splits_for_sens:
    ax.plot(bin_range, [sens_df.loc[b, split_name] for b in bin_range],
            marker="o", label=split_name, alpha=0.85)
ax.axvline(x=PSI_BINS, color="gray", linestyle="--", linewidth=1.0, label=f"PSI_BINS={PSI_BINS}")
ax.set_xlabel("Number of PSI bins")
ax.set_ylabel("PSI value")
ax.set_xticks(bin_range)
ax.set_title("PSI sensitivity to bin count", fontsize=10)
ax.legend(fontsize=8, loc="upper left")
ax.grid(alpha=0.3, linestyle="--")
plt.tight_layout()
plt.show()

print(f"Conclusion: PSI_BINS={PSI_BINS} is in the stable region.")
print("Split ordering is preserved across all bin counts (ordering rho >= 0.90).")
print("Bins >= 7 may inflate values for quantised propensity distributions;")

# ── §A1 CSV export ────────────────────────────────────────────────────────
# Persist the bin-sensitivity matrix as CSV so scripts/build_thesis_tables.py
# can read it and emit the appendix LaTeX table.
_psi_prop_csv = ARTIFACT_DIR / PRIMARY_VARIANT / "psi_bin_sensitivity_propensity.csv"
sens_df.to_csv(_psi_prop_csv)
print(f"Saved propensity PSI sensitivity → {_psi_prop_csv}")

In [ ]:
# ── §A2  PSI sensitivity to bin count (AUC) ──────────────────────────────
# Same analysis as §17b but applied to the modelPerformance (AUC) distribution.
# Validates that PSI_BINS=5 is also appropriate for the δ_m proxy.
# Pega AUC values are discretised (typically a small set of distinct floats),
# so the same empty-bin inflation risk exists as for propensity scores.

bin_range = [3, 4, 5, 6, 7, 10]

auc_splits_for_sens = {
    f"{PRIMARY_VARIANT} temporal": (
        meta.loc[mask_early, "modelPerformance"].dropna(),
        meta.loc[mask_late,  "modelPerformance"].dropna(),
    )
}
for i in range(len(top_routes) - 1):
    r1, r2 = top_routes[i], top_routes[i + 1]
    auc_splits_for_sens[f"{r1} vs {r2}"] = (
        meta.loc[route_masks[r1], "modelPerformance"].dropna(),
        meta.loc[route_masks[r2], "modelPerformance"].dropna(),
    )

# Culture split (added for parity with the main attribution analysis)
_culture_col = "CustBookedFlight.BookingData.CultureCode"
_mask_fr = df[_culture_col] == "fr-FR"
_mask_nl = df[_culture_col] == "nl-NL"
if _mask_fr.any() and _mask_nl.any():
    auc_splits_for_sens["fr-FR vs nl-NL"] = (
        meta.loc[_mask_fr, "modelPerformance"].dropna(),
        meta.loc[_mask_nl, "modelPerformance"].dropna(),
    )

auc_sens_rows = []
for b in bin_range:
    row = {"bins": b}
    for split_name, (s_a, s_b) in auc_splits_for_sens.items():
        psi_v, _ = psi(s_a, s_b, bins=b)
        row[split_name] = round(psi_v, 4)
    auc_sens_rows.append(row)

auc_sens_df = pd.DataFrame(auc_sens_rows).set_index("bins")

print("AUC PSI values per split at each bin count:")
display(
    auc_sens_df.style
    .format("{:.4f}")
    .background_gradient(cmap="YlOrRd", vmin=0, vmax=1.0, axis=None)
    .highlight_max(color="#f8d7da", axis=0)
)

ref_ranks_auc = auc_sens_df.loc[PSI_BINS].rank()
print(f"Spearman rho of split ordering (vs PSI_BINS={PSI_BINS} reference):")
for b in bin_range:
    rho, _ = _sp_sens(ref_ranks_auc, auc_sens_df.loc[b].rank())
    marker = " <-- chosen" if b == PSI_BINS else ""
    print(f"  bins={b:2d}  ordering rho={rho:.4f}{marker}")

fig, ax = plt.subplots(figsize=(8, 3.5))
for split_name in auc_splits_for_sens:
    ax.plot(bin_range, [auc_sens_df.loc[b, split_name] for b in bin_range],
            marker="o", label=split_name, alpha=0.85)
ax.axvline(x=PSI_BINS, color="gray", linestyle="--", linewidth=1.0, label=f"PSI_BINS={PSI_BINS}")
ax.set_xlabel("Number of PSI bins")
ax.set_ylabel("AUC PSI value")
ax.set_xticks(bin_range)
ax.set_title("AUC PSI sensitivity to bin count", fontsize=10)
ax.legend(fontsize=8, loc="upper left")
ax.grid(alpha=0.3, linestyle="--")
plt.tight_layout()
plt.show()

print(f"Conclusion: PSI_BINS={PSI_BINS} applied to AUC distribution.")
print("Check whether ordering is stable and whether higher bin counts inflate values.")

# ── §A2 CSV export ────────────────────────────────────────────────────────
_psi_auc_csv = ARTIFACT_DIR / PRIMARY_VARIANT / "psi_bin_sensitivity_auc.csv"
auc_sens_df.to_csv(_psi_auc_csv)
print(f"Saved AUC PSI sensitivity → {_psi_auc_csv}")

In [ ]:
# ── §A3  δ_m proxies side-by-side (AUC PSI vs |Δμ| vs KS) ────────────────
# Compare three delta_m metrics side by side: AUC PSI (current), mean AUC
# difference, and KS statistic. PSI is bin-sensitive (see 17c); mean diff
# and KS are bin-free. Use this to validate or replace AUC PSI as delta_m proxy.

from scipy.stats import ks_2samp

def _auc_metrics(s_a, s_b):
    psi_v, _  = psi(s_a, s_b, bins=PSI_BINS)
    mean_diff = abs(float(s_b.mean()) - float(s_a.mean()))
    ks_stat, ks_p = ks_2samp(s_a, s_b)
    return round(psi_v, 4), round(mean_diff, 4), round(ks_stat, 4), round(ks_p, 4)

cmp_rows = []

_pe = meta.loc[mask_early, "modelPerformance"].dropna()
_pl = meta.loc[mask_late,  "modelPerformance"].dropna()
_psi, _md, _ks, _ksp = _auc_metrics(_pe, _pl)
cmp_rows.append({"split": f"{PRIMARY_VARIANT} temporal", "AUC PSI": _psi, "delta_mu_AUC": _md, "KS": _ks, "KS p": _ksp})

for i in range(len(top_routes) - 1):
    r1, r2 = top_routes[i], top_routes[i + 1]
    _p1 = meta.loc[route_masks[r1], "modelPerformance"].dropna()
    _p2 = meta.loc[route_masks[r2], "modelPerformance"].dropna()
    _psi, _md, _ks, _ksp = _auc_metrics(_p1, _p2)
    cmp_rows.append({"split": f"{r1} vs {r2}", "AUC PSI": _psi, "delta_mu_AUC": _md, "KS": _ks, "KS p": _ksp})

cmp_df = pd.DataFrame(cmp_rows).set_index("split")

print("AUC model churn metrics comparison (delta_m proxies):")
print("  AUC PSI      = PSI on modelPerformance distribution (bins=5, bin-sensitive)")
print("  delta_mu_AUC = |mean AUC late - mean AUC early| (bin-free)")
print("  KS           = Kolmogorov-Smirnov statistic on AUC distributions (bin-free)")
print("  KS p         = p-value (< 0.05 = distributions significantly different)")
print()
display(
    cmp_df.style
    .format({"AUC PSI": "{:.4f}", "delta_mu_AUC": "{:.4f}", "KS": "{:.4f}", "KS p": "{:.4f}"})
    .background_gradient(subset=["AUC PSI"],      cmap="YlOrRd", vmin=0,    vmax=15.0)
    .background_gradient(subset=["delta_mu_AUC"], cmap="YlOrRd", vmin=0,    vmax=0.10)
    .background_gradient(subset=["KS"],           cmap="YlOrRd", vmin=0,    vmax=1.0)
    .background_gradient(subset=["KS p"],         cmap="RdYlGn", vmin=0,    vmax=0.10)
)

## §A4  Jaccard threshold sensitivity (δ_m cutoff)

The δ_m metric is the Jaccard overlap on the set of `modelVersion` IDs active in each
sub-population. Smaller `J_v` = fewer shared model snapshots = higher model churn.

**Threshold question.** We need a cutoff `J_v ≤ τ` that separates "real model churn" from
"no churn". This appendix shows the cutoff is robust because the observed `J_v` values are
strongly bimodal:

- **Temporal splits** (early vs late, by design crossing many model updates) → `J_v ≈ 0`.
- **Same-window splits** (route pairs, culture pairs, all inside the same time window) →
  `J_v ≈ 0.5`. This is *not* real churn — Pega's `modelVersion` UUID changes on every
  micro-update of the online Naive Bayes, even when the parameters barely move. With
  ~1,500 unique UUIDs across 31k L5B15 rows (≈ 1 new UUID per 21 rows), two same-week
  subgroups naturally see ~half their snapshot UUIDs unique to one side.

So the threshold can sit anywhere in the empty gap between ~0.01 and ~0.48. We chose
`τ = 0.10` — well above the temporal floor (gives margin) and well below the in-window
ceiling (avoids false positives). The check below shows classifications are insensitive to
any choice in [0.05, 0.40].

In [ ]:
# ── §A4  Jaccard threshold sensitivity ────────────────────────────────────
# Two checks:
#   1. Strip plot: where do the observed J_v values actually sit on the [0,1] line?
#   2. Threshold sweep: for τ ∈ {0.01, 0.05, 0.10, 0.25, 0.40, 0.50, 0.70},
#      how many splits get flagged as model churn? Identify the safe band.

import numpy as np
import matplotlib.pyplot as plt

j_values = pd.Series(jaccard_v_lookup, name="J_v").sort_values()
print("Observed J_v values per split (sorted):")
display(j_values.to_frame().style.format({"J_v": "{:.4f}"}))

# Gap between the largest temporal-style value and the smallest in-window value
# (assuming temporal sits near 0, in-window near 0.5)
in_window_min = j_values[j_values > 0.1].min() if (j_values > 0.1).any() else float("nan")
temporal_max  = j_values[j_values < 0.1].max() if (j_values < 0.1).any() else float("nan")
print(f"\nBimodality gap: temporal cluster max = {temporal_max:.4f}, "
      f"in-window cluster min = {in_window_min:.4f}")
print(f"  → any threshold in ({temporal_max:.2f}, {in_window_min:.2f}) gives identical classifications.")

# ── Strip plot: J_v values on the real line ───────────────────────────────
fig, ax = plt.subplots(figsize=(9, 2.6))
xs = j_values.values
ax.scatter(xs, np.zeros_like(xs) + 0.05, s=110, color="steelblue", alpha=0.85, zorder=3)
for split_name, jv in j_values.items():
    ax.annotate(split_name, (jv, 0.05),
                xytext=(0, 14), textcoords="offset points",
                ha="center", fontsize=8, rotation=25)

# Candidate thresholds
for tau, color, lbl in [(0.05, "#cccccc", "0.05"),
                        (0.10, "darkorange", "0.10 (chosen)"),
                        (0.25, "#cccccc", "0.25"),
                        (0.50, "#999999", "0.50 (old)")]:
    ax.axvline(tau, color=color, linestyle="--", linewidth=1.0, alpha=0.85)
    ax.text(tau, -0.06, f"τ={lbl}", ha="center", va="top", fontsize=8, color=color)

ax.set_xlim(-0.02, 1.0)
ax.set_ylim(-0.18, 0.32)
ax.set_yticks([])
ax.set_xlabel("J_v (Jaccard overlap on modelVersion sets)")
ax.set_title("§A4 · J_v values per split — strongly bimodal; threshold lives in the empty gap",
             fontsize=10)
for spine in ("left", "right", "top"):
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()

# ── Threshold sweep: classification under varying τ ───────────────────────
TAUS = [0.01, 0.05, 0.10, 0.25, 0.40, 0.50, 0.70]

def _classify_at(ks_d, j_v, tau_j):
    d_sig = ks_d >= KS_PROP_THRESHOLD
    m_sig = j_v  <= tau_j
    if not d_sig and not m_sig: return "Explainer"
    if d_sig and not m_sig:     return "δ_d"
    if m_sig and not d_sig:     return "δ_m"
    return "Both"

sweep_rows = []
for split in attr_df.index:
    row = {"split": split, "J_v": jaccard_v_lookup[split], "KS_prop": ks_prop_lookup[split]}
    for tau in TAUS:
        row[f"τ={tau:.2f}"] = _classify_at(ks_prop_lookup[split], jaccard_v_lookup[split], tau)
    sweep_rows.append(row)
sweep_df = pd.DataFrame(sweep_rows).set_index("split")

print("\nClassification under different J_v thresholds:")
display(
    sweep_df.style.format({"J_v": "{:.4f}", "KS_prop": "{:.4f}"})
    .set_properties(subset=[f"τ={t:.2f}" for t in TAUS], **{"text-align": "center"})
    .apply(lambda col: [
        ("background-color: #fff3cd; font-weight: bold" if v == "Both"
         else "background-color: #cce5ff" if v == "δ_m"
         else "background-color: #ffe0b2" if v == "δ_d"
         else "background-color: #dff0d8" if v == "Explainer"
         else "") for v in col
    ], subset=[f"τ={t:.2f}" for t in TAUS])
)

# Identify the τ range over which classifications are stable
labels_at_tau = {t: tuple(sweep_df[f"τ={t:.2f}"]) for t in TAUS}
stable_band = [t for t in TAUS if labels_at_tau[t] == labels_at_tau[JACCARD_V_THRESHOLD]]
print(f"\nClassifications identical to chosen τ={JACCARD_V_THRESHOLD} for τ ∈ {stable_band}")
print(f"Chosen τ={JACCARD_V_THRESHOLD} sits comfortably inside the stable band.")